In [ ]:
#compute optimal thershold race

In [1]:
import os
import sys
import json
import copy
import socket
import getpass
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
sys.path.append('../')
import pickle
import warnings
from sklearn.metrics import recall_score, matthews_corrcoef, roc_auc_score, f1_score
from collections import defaultdict
import json

warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

def flatten(l):
    return [item for sublist in l for item in sublist]

def compute_opt_thres(target, pred):
    opt_thres = 0
    opt_f1 = 0
    for i in np.arange(0.001, 0.999, 0.001):
        f1 = f1_score(target, pred >= i)
        if f1 >= opt_f1:
            opt_thres = i
            opt_f1 = f1
    return opt_thres

plt.rcParams.update({'font.size': 20})

In [2]:
#root_dir = Path(f'/path/to/your/root')
root_dir = Path(f'/home/lchanch/initial_training/output_sweep_12/grid_race_mimic_12')

In [3]:
if Path('opt_thres.pkl').is_file():
    already_computed = set(pickle.load(Path('opt_thres.pkl').open('rb')).keys())
else:
    already_computed = set()

In [4]:
results = []
for i in tqdm(root_dir.glob('**/done')):
    args = json.load((i.parent/'args.json').open('r'))
    if (args['dataset'][0], args['task'], args['attr'], args['algorithm']) in already_computed:
        continue
    
    final_res = pickle.load((i.parent/'final_results.pkl').open('rb'))
    
    ssets = ['va', 'te', 'MIMIC-sex-te', 'CheXpert-sex-te']
  
    for sset in ssets:
        if sset in final_res:
            args[f'{sset}_auroc'] = final_res[sset]['overall']['AUROC']
            if sset == 'va':
                args[f'{sset}_min_attr_auroc'] = final_res[sset]['min_attr']['AUROC']
    args['va_y'] = final_res['va']['y']
    args['va_preds'] = final_res['va']['preds']
    
    results.append(args)
df = pd.DataFrame(results)

136it [00:01, 117.67it/s]


In [5]:
df['dataset'] = df['dataset'].apply(lambda x: x[0])

In [6]:
df.shape

(136, 33)

In [7]:
df.head()

,store_name,dataset,task,attr,group_def,algorithm,output_dir,data_dir,hparams,hparams_seed,...,checkpoint_freq,skip_model_save,debug,image_arch,aug,va_auroc,va_min_attr_auroc,te_auroc,va_y,va_preds
0,c60d92463516d8ba5a25fab8dfc14d70,MIMIC,Cardiomegaly,ethnicity,group,ERM,/home/lchanch/initial_training/output_sweep_12...,/home/lchanch/initial_training/image_df,{},7,...,1000,False,False,densenet_sup_in1k,basic2,0.807013,0.799988,0.804031,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, ...","[0.07763106, 0.057760686, 0.05734263, 0.130005..."
1,359f2b34389f45be0d316685d88d17ed,MIMIC,Pleural Effusion,ethnicity,group,ERM,/home/lchanch/initial_training/output_sweep_12...,/home/lchanch/initial_training/image_df,{},0,...,1000,False,False,densenet_sup_in1k,basic2,0.894821,0.884731,0.898042,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.10676424, 0.12600476, 0.011332679, 0.022382..."
2,08a0366e16736f3922816d9e98cb1ec6,MIMIC,Pneumothorax,ethnicity,group,ERM,/home/lchanch/initial_training/output_sweep_12...,/home/lchanch/initial_training/image_df,{},9,...,1000,False,False,densenet_sup_in1k,basic2,0.853951,0.813886,0.852707,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.012010355, 0.0014049882, 0.007486078, 0.003..."
3,d6036a621a8e0f41e0a49d81826bcac7,MIMIC,No Finding,ethnicity,group,ERM,/home/lchanch/initial_training/output_sweep_12...,/home/lchanch/initial_training/image_df,{},6,...,1000,False,False,densenet_sup_in1k,basic2,0.844407,0.837862,0.846412,"[0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, ...","[0.18789078, 0.50233406, 0.32918775, 0.6254154..."
4,6a9942b80637c37abe19ca8a3bcb575b,MIMIC,Pneumothorax,ethnicity,group,ERM,/home/lchanch/initial_training/output_sweep_12...,/home/lchanch/initial_training/image_df,{},9,...,1000,False,False,densenet_sup_in1k,basic2,0.841966,0.787466,0.840835,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.020646162, 0.003557798, 0.03180448, 0.00846..."


In [ ]:
#optimal_thresholds

In [ ]:
#best models según va_min_attr_auroc

In [8]:
best_models = df.groupby(['dataset', 'task', 'attr', 'algorithm']).apply(lambda x: x.loc[x['va_min_attr_auroc'].idxmax()])

/tmp/ipykernel_2998920/3613221529.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  best_models = df.groupby(['dataset', 'task', 'attr', 'algorithm']).apply(lambda x: x.loc[x['va_min_attr_auroc'].idxmax()])


In [9]:
best_models

store_name  \
dataset task             attr      algorithm                                     
MIMIC   Cardiomegaly     ethnicity ERM        f33c28fbb7c9261fd5bfacaaa32e5011   
        No Finding       ethnicity ERM        9dc25811e744369d2707c706df7f4068   
        Pleural Effusion ethnicity ERM        70d3048b4e48381e174cd9189833f453   
        Pneumothorax     ethnicity ERM        18bb25571076cd85c2094f5f56772584   

                                             dataset              task  \
dataset task             attr      algorithm                             
MIMIC   Cardiomegaly     ethnicity ERM         MIMIC      Cardiomegaly   
        No Finding       ethnicity ERM         MIMIC        No Finding   
        Pleural Effusion ethnicity ERM         MIMIC  Pleural Effusion   
        Pneumothorax     ethnicity ERM         MIMIC      Pneumothorax   

                                                   attr group_def algorithm  \
dataset task             attr      algorithm                                  
MIMIC   Cardiomegaly     ethnicity ERM        ethnicity     group       ERM   
        No Finding       ethnicity ERM        ethnicity     group       ERM   
        Pleural Effusion ethnicity ERM        ethnicity     group       ERM   
        Pneumothorax     ethnicity ERM        ethnicity     group       ERM   

                                                                                     output_dir  \
dataset task             attr      algorithm                                                      
MIMIC   Cardiomegaly     ethnicity ERM        /home/lchanch/initial_training/output_sweep_12...   
        No Finding       ethnicity ERM        /home/lchanch/initial_training/output_sweep_12...   
        Pleural Effusion ethnicity ERM        /home/lchanch/initial_training/output_sweep_12...   
        Pneumothorax     ethnicity ERM        /home/lchanch/initial_training/output_sweep_12...   

                                                                             data_dir  \
dataset task             attr      algorithm                                            
MIMIC   Cardiomegaly     ethnicity ERM        /home/lchanch/initial_training/image_df   
        No Finding       ethnicity ERM        /home/lchanch/initial_training/image_df   
        Pleural Effusion ethnicity ERM        /home/lchanch/initial_training/image_df   
        Pneumothorax     ethnicity ERM        /home/lchanch/initial_training/image_df   

                                             hparams  hparams_seed  ...  \
dataset task             attr      algorithm                        ...   
MIMIC   Cardiomegaly     ethnicity ERM            {}             1  ...   
        No Finding       ethnicity ERM            {}             1  ...   
        Pleural Effusion ethnicity ERM            {}             9  ...   
        Pneumothorax     ethnicity ERM            {}             1  ...   

                                              checkpoint_freq skip_model_save  \
dataset task             attr      algorithm                                    
MIMIC   Cardiomegaly     ethnicity ERM                   1000           False   
        No Finding       ethnicity ERM                   1000           False   
        Pleural Effusion ethnicity ERM                   1000           False   
        Pneumothorax     ethnicity ERM                   1000           False   

                                              debug         image_arch  \
dataset task             attr      algorithm                             
MIMIC   Cardiomegaly     ethnicity ERM        False  densenet_sup_in1k   
        No Finding       ethnicity ERM        False  densenet_sup_in1k   
        Pleural Effusion ethnicity ERM        False  densenet_sup_in1k   
        Pneumothorax     ethnicity ERM        False  densenet_sup_in1k   

                                                 aug  va_auroc  \
dataset task             attr      algorithm                     
MIMIC   Ca

In [10]:
opt_thres = {}
for idx, row in tqdm(best_models.iterrows(), total = len(best_models)):
    dataset, task, attr, algorithm = idx
#     if dataset not in opt_thres:
#         opt_thres[dataset] = {}
    opt_thres[(dataset, task, attr, algorithm)] = np.round(compute_opt_thres(row['va_y'], row['va_preds']), 3)

100%|██████████| 4/4 [00:31<00:00,  7.91s/it]


In [11]:
if Path('opt_thres.pkl').is_file():
    old_file = pickle.load(Path('opt_thres.pkl').open('rb'))
else:
    old_file = {}

In [12]:
opt_thres = {**old_file, **opt_thres}

In [13]:
opt_thres

{('MIMIC', 'Cardiomegaly', 'ethnicity', 'ERM'): 0.245,
 ('MIMIC', 'No Finding', 'ethnicity', 'ERM'): 0.425,
 ('MIMIC', 'Pleural Effusion', 'ethnicity', 'ERM'): 0.385,
 ('MIMIC', 'Pneumothorax', 'ethnicity', 'ERM'): 0.17}

In [15]:
#se observa que los opt thersholds son los mismos excepto para pleural effusoin, que en el de 3 semillas era 0.37

In [14]:
pickle.dump(opt_thres, open('opt_thres_race_mimic_12.pkl', 'wb'))

In [16]:
best_models['hparams_seed']

dataset  task              attr       algorithm
MIMIC    Cardiomegaly      ethnicity  ERM          1
         No Finding        ethnicity  ERM          1
         Pleural Effusion  ethnicity  ERM          9
         Pneumothorax      ethnicity  ERM          1
Name: hparams_seed, dtype: int64

In [20]:
best_models[['store_name','hparams_seed']]

store_name  \
dataset task             attr      algorithm                                     
MIMIC   Cardiomegaly     ethnicity ERM        f33c28fbb7c9261fd5bfacaaa32e5011   
        No Finding       ethnicity ERM        9dc25811e744369d2707c706df7f4068   
        Pleural Effusion ethnicity ERM        70d3048b4e48381e174cd9189833f453   
        Pneumothorax     ethnicity ERM        18bb25571076cd85c2094f5f56772584   

                                              hparams_seed  
dataset task             attr      algorithm                
MIMIC   Cardiomegaly     ethnicity ERM                   1  
        No Finding       ethnicity ERM                   1  
        Pleural Effusion ethnicity ERM                   9  
        Pneumothorax     ethnicity ERM                   1